# Archive Papers Viewer

This notebook provides tools to view and analyze papers in the Station's archive, with proper ranking by review scores and formatted display. **Enhanced with appendix/addendum viewing capabilities.**

In [ ]:
import yaml
import os
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
from datetime import datetime

In [ ]:
class ArchiveViewer:
    """Class to manage and display archive papers from the Station."""
    
    def __init__(self, archive_path: str = "."):
        """Initialize the archive viewer.
        
        Args:
            archive_path: Path to the archive directory containing YAML files
        """
        self.archive_path = Path(archive_path)
        self.papers = {}
        self.load_papers()
    
    def load_papers(self):
        """Load all archive YAML files from the directory."""
        yaml_files = list(self.archive_path.glob("archive_*.yaml"))
        
        # Sort files numerically by the ID number
        yaml_files.sort(key=lambda x: int(x.stem.split('_')[1]))
        
        for yaml_file in yaml_files:
            try:
                with open(yaml_file, 'r', encoding='utf-8') as f:
                    data = yaml.safe_load(f)
                    if data:
                        # Extract paper ID from filename
                        paper_id = int(yaml_file.stem.split('_')[1])
                        data['paper_id'] = paper_id
                        
                        # Extract score from review if exists
                        score = self.extract_score(data)
                        data['score'] = score
                        
                        self.papers[paper_id] = data
            except Exception as e:
                print(f"Error loading {yaml_file}: {e}")
    
    def extract_score(self, paper_data: Dict) -> Optional[float]:
        """Extract the review score from paper data.
        
        Args:
            paper_data: Dictionary containing paper data
            
        Returns:
            Score as float if found, None otherwise
        """
        if 'messages' not in paper_data:
            return None
            
        for message in paper_data.get('messages', []):
            if message.get('author_name') == 'Archive Review System':
                content = message.get('content', '')
                # Look for score pattern like "Score: 6/10" or "**Score:** 6/10"
                score_match = re.search(r'\*?\*?Score:?\*?\*?\s*(\d+(?:\.\d+)?)/10', content, re.IGNORECASE)
                if score_match:
                    return float(score_match.group(1))
        return None
    
    def get_ranked_papers(self) -> List[Tuple[int, Dict]]:
        """Get papers ranked by score from highest to lowest.
        
        Returns:
            List of tuples (paper_id, paper_data) sorted by score
        """
        papers_with_scores = [(pid, pdata) for pid, pdata in self.papers.items() 
                               if pdata.get('score') is not None]
        papers_without_scores = [(pid, pdata) for pid, pdata in self.papers.items() 
                                 if pdata.get('score') is None]
        
        # Sort papers with scores by score (descending), then by ID for ties
        papers_with_scores.sort(key=lambda x: (-x[1]['score'], x[0]))
        
        # Sort papers without scores by ID (numerically)
        papers_without_scores.sort(key=lambda x: x[0])
        
        return papers_with_scores + papers_without_scores
    
    def format_content(self, content: str) -> str:
        """Format content for proper markdown display.
        
        Args:
            content: Raw content string
            
        Returns:
            Formatted content string
        """
        # Handle escaped newlines and ensure proper markdown formatting
        if content:
            # Replace literal \n with actual newlines
            content = content.replace('\\n', '\n')
            # Ensure proper spacing for markdown headers
            content = re.sub(r'(#+)([^\s#])', r'\1 \2', content)
        return content
    
    def get_paper_summary(self, paper_id: int) -> Dict:
        """Get a summary of a paper.
        
        Args:
            paper_id: ID of the paper
            
        Returns:
            Dictionary with paper summary information
        """
        paper = self.papers.get(paper_id, {})
        
        return {
            'id': paper_id,
            'title': paper.get('title', 'Untitled'),
            'author': paper.get('author_name', 'Unknown'),
            'score': paper.get('score'),
            'tick': paper.get('created_at_tick', 0),
            'abstract': paper.get('abstract', 'No abstract available'),
            'word_count': paper.get('word_count_total', 0)
        }
    
    def get_paper_content(self, paper_id: int) -> Optional[str]:
        """Get the main content of a paper.
        
        Args:
            paper_id: ID of the paper
            
        Returns:
            Paper content as string, or None if not found
        """
        paper = self.papers.get(paper_id)
        if not paper or 'messages' not in paper:
            return None
            
        for message in paper['messages']:
            if message.get('author_name') != 'Archive Review System':
                content = message.get('content', '')
                return self.format_content(content)
        return None
    
    def get_review_content(self, paper_id: int) -> Optional[str]:
        """Get the review content of a paper.
        
        Args:
            paper_id: ID of the paper
            
        Returns:
            Review content as string, or None if not found
        """
        paper = self.papers.get(paper_id)
        if not paper or 'messages' not in paper:
            return None
            
        for message in paper['messages']:
            if message.get('author_name') == 'Archive Review System':
                content = message.get('content', '')
                return self.format_content(content)
        return None
    
    def get_appendices(self, paper_id: int) -> List[Dict]:
        """Get all appendix/addendum content of a paper.
        
        Args:
            paper_id: ID of the paper
            
        Returns:
            List of appendix/addendum messages with title and content
        """
        paper = self.papers.get(paper_id)
        if not paper or 'messages' not in paper:
            return []
        
        appendices = []
        messages = paper['messages']
        
        # Skip the first message (main paper) and review messages
        # Look for additional messages that are appendices/addenda
        main_found = False
        for message in messages:
            author = message.get('author_name', '')
            title = message.get('title', '')
            content = message.get('content', '')
            
            # Skip review system messages
            if author == 'Archive Review System':
                continue
            
            # If this is the first non-review message, it's the main paper
            if not main_found:
                main_found = True
                continue
            
            # Any additional non-review messages are appendices
            # Check if title suggests it's an appendix/addendum
            # is_appendix = any(keyword in title.lower() for keyword in [
            #     'addendum', 'appendix', 'supplement', 'correction', 'update', 
            #     'results', 'analysis', 'follow-up', 'erratum'
            # ]) or 'v' in title.lower()  # Often versioned like "v1", "v2"
            is_appendix = True
            
            if is_appendix or main_found:  # Include all additional messages
                appendices.append({
                    'title': title or f"Appendix {len(appendices) + 1}",
                    'content': self.format_content(content),
                    'author': author,
                    'tick': message.get('posted_at_tick', 0),
                    'word_count': message.get('word_count', 0)
                })
        
        return appendices
    
    def has_appendices(self, paper_id: int) -> bool:
        """Check if a paper has appendices/addenda.
        
        Args:
            paper_id: ID of the paper
            
        Returns:
            True if paper has appendices, False otherwise
        """
        return len(self.get_appendices(paper_id)) > 0

In [ ]:
# Initialize the archive viewer
viewer = ArchiveViewer('.')
print(f"Loaded {len(viewer.papers)} papers from the archive")

## Papers Ranked by Score

Display all papers ranked from highest to lowest score:

In [ ]:
# Display ranked papers table
ranked_papers = viewer.get_ranked_papers()

# Create HTML table for better formatting
html_content = """
<style>
    .paper-table {
        border-collapse: collapse;
        width: 100%;
        margin: 20px 0;
    }
    .paper-table th, .paper-table td {
        border: 1px solid #ddd;
        padding: 12px;
        text-align: left;
    }
    .paper-table th {
        background-color: #4CAF50;
        color: white;
    }
    .paper-table tr:nth-child(even) {
        background-color: #f2f2f2;
    }
    .paper-table tr:hover {
        background-color: #ddd;
    }
    .score-high { color: #2e7d32; font-weight: bold; }
    .score-medium { color: #f57c00; font-weight: bold; }
    .score-low { color: #d32f2f; font-weight: bold; }
    .no-score { color: #9e9e9e; font-style: italic; }
    .has-appendix { color: #1976d2; font-weight: bold; }
</style>

<table class="paper-table">
<thead>
<tr>
    <th>Rank</th>
    <th>ID</th>
    <th>Score</th>
    <th>Title</th>
    <th>Author</th>
    <th>Tick</th>
    <th>Words</th>
    <th>Appendix</th>
</tr>
</thead>
<tbody>
"""

for rank, (pid, paper) in enumerate(ranked_papers, 1):
    summary = viewer.get_paper_summary(pid)
    
    # Format score with color coding
    if summary['score'] is not None:
        score = summary['score']
        if score >= 9:
            score_class = 'score-high'
        elif score >= 8:
            score_class = 'score-medium'
        else:
            score_class = 'score-low'
        score_str = f'<span class="{score_class}">{score:.1f}/10</span>'
    else:
        score_str = '<span class="no-score">No review</span>'
    
    # Check for appendices
    has_appendix = viewer.has_appendices(pid)
    appendix_count = len(viewer.get_appendices(pid))
    if has_appendix:
        appendix_str = f'<span class="has-appendix">✓ ({appendix_count})</span>'
    else:
        appendix_str = '–'
    
    # Truncate long titles
    title = summary['title']
    if len(title) > 60:
        title = title[:57] + '...'
    
    html_content += f"""
<tr>
    <td>{rank}</td>
    <td>{summary['id']}</td>
    <td>{score_str}</td>
    <td>{title}</td>
    <td>{summary['author']}</td>
    <td>{summary['tick']}</td>
    <td>{summary['word_count']}</td>
    <td>{appendix_str}</td>
</tr>
"""

html_content += """
</tbody>
</table>
"""

display(HTML(html_content))

## Interactive Paper Viewer

Select a paper from the dropdown to view its abstract, content, review, and appendices/addenda:

In [ ]:
# Create interactive viewer with dropdown and ID input - with background preloading
import asyncio
import threading
import time

class InteractivePaperViewer:
    def __init__(self, viewer):
        self.viewer = viewer
        self.current_paper_id = None
        self.cache = {}  # Cache for preloaded papers
        self.preload_queue = []  # Queue of papers to preload
        self.preload_stop = False  # Flag to stop preloading
        self.preload_thread = None
        self.setup_widgets()
    
    def setup_widgets(self):
        """Setup the interactive widgets."""
        print("Loading papers list...")
        
        # Get ranked papers for dropdown
        ranked_papers = self.viewer.get_ranked_papers()
        
        # Create dropdown options more efficiently
        options = []
        self.id_to_dropdown_value = {}  # Map paper IDs to dropdown values
        self.paper_ids = []  # List of all paper IDs in ranked order
        
        for rank, (pid, paper) in enumerate(ranked_papers, 1):
            # Only access the fields we need for the dropdown
            title = paper.get('title', 'Untitled')
            score = paper.get('score')
            score_str = f"{score:.1f}" if score else "N/A"
            
            # Add appendix indicator
            has_appendix = self.viewer.has_appendices(pid)
            appendix_indicator = " [+A]" if has_appendix else ""
            
            if len(title) > 35:  # Shortened to make room for appendix indicator
                title = title[:32] + '...'
            label = f"#{rank} | ID:{pid} | Score:{score_str} | {title}{appendix_indicator}"
            options.append((label, pid))
            self.id_to_dropdown_value[pid] = pid
            self.paper_ids.append(pid)
        
        # Create dropdown widget
        self.dropdown = widgets.Dropdown(
            options=options,
            description='Select Paper:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='100%')
        )
        
        # Create ID input widget
        self.id_input = widgets.IntText(
            value=1,
            description='Paper ID:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='150px')
        )
        
        # Create Go button for ID input
        self.go_button = widgets.Button(
            description='Go to ID',
            button_style='info',
            layout=widgets.Layout(width='100px')
        )
        
        # Create status label for preloading
        self.status_label = widgets.HTML(
            value='<span style="color: gray; font-size: 12px;">Ready</span>'
        )
        
        # Create output area
        self.output_area = widgets.Output()
        
        # Set up event handlers
        self.dropdown.observe(self.on_dropdown_change, names='value')
        self.go_button.on_click(self.on_go_clicked)
        self.id_input.observe(self.on_id_input_submit, names='value')
        
        # Display widgets
        display(widgets.VBox([
            self.dropdown,
            widgets.HBox([
                widgets.HTML('<b>Or select by ID:</b>'),
                self.id_input,
                self.go_button,
                self.status_label
            ]),
            widgets.HTML('<hr style="margin: 20px 0;">'),
            self.output_area
        ]))
        
        print(f"Ready! {len(options)} papers available. [+A] indicates papers with appendices.")
        
        # Show first paper by default and start preloading
        if options:
            self.display_paper(options[0][1])
            self.start_preloading(options[0][1])
    
    def preload_paper(self, paper_id):
        """Preload a single paper's content into cache."""
        if paper_id not in self.cache:
            try:
                summary = self.viewer.get_paper_summary(paper_id)
                content = self.viewer.get_paper_content(paper_id)
                review = self.viewer.get_review_content(paper_id)
                appendices = self.viewer.get_appendices(paper_id)
                
                self.cache[paper_id] = {
                    'summary': summary,
                    'content': content,
                    'review': review,
                    'appendices': appendices
                }
            except Exception as e:
                print(f"Error preloading paper {paper_id}: {e}")
    
    def preload_worker(self):
        """Background worker to preload papers."""
        while not self.preload_stop and self.preload_queue:
            paper_id = self.preload_queue.pop(0)
            if paper_id not in self.cache:
                self.preload_paper(paper_id)
                # Update status
                cached_count = len(self.cache)
                total_count = len(self.paper_ids)
                self.status_label.value = f'<span style="color: green; font-size: 12px;">Cached: {cached_count}/{total_count}</span>'
                
                # Small delay to avoid blocking
                time.sleep(0.1)
        
        if not self.preload_stop:
            self.status_label.value = '<span style="color: blue; font-size: 12px;">All papers cached!</span>'
    
    def start_preloading(self, current_paper_id):
        """Start preloading papers in background, prioritizing nearby papers."""
        # Stop any existing preloading
        self.stop_preloading()
        
        # Create new preload queue
        self.preload_queue = []
        
        # Find current paper index
        try:
            current_index = self.paper_ids.index(current_paper_id)
        except ValueError:
            current_index = 0
        
        # Add papers to queue, prioritizing nearby papers
        # First add papers after current one
        for i in range(current_index + 1, len(self.paper_ids)):
            self.preload_queue.append(self.paper_ids[i])
        
        # Then add papers before current one
        for i in range(0, current_index):
            self.preload_queue.append(self.paper_ids[i])
        
        # Start preloading thread
        self.preload_stop = False
        self.preload_thread = threading.Thread(target=self.preload_worker, daemon=True)
        self.preload_thread.start()
    
    def stop_preloading(self):
        """Stop the background preloading."""
        self.preload_stop = True
        if self.preload_thread and self.preload_thread.is_alive():
            self.preload_thread.join(timeout=0.5)
    
    def on_dropdown_change(self, change):
        """Handle dropdown selection change."""
        paper_id = change['new']
        if paper_id != self.current_paper_id:
            self.display_paper(paper_id)
            self.start_preloading(paper_id)
    
    def on_go_clicked(self, b):
        """Handle Go button click."""
        self.go_to_paper_id(self.id_input.value)
    
    def on_id_input_submit(self, change):
        """Handle Enter key in ID input (if user presses Enter)."""
        # This will trigger when value changes, but we'll use the button for explicit action
        pass
    
    def go_to_paper_id(self, paper_id):
        """Navigate to a specific paper by ID."""
        if paper_id in self.viewer.papers:
            # Update dropdown to match the selected ID
            if paper_id in self.id_to_dropdown_value:
                self.dropdown.value = self.id_to_dropdown_value[paper_id]
            # Display the paper (dropdown change will trigger this too, but we handle duplicates)
            self.display_paper(paper_id)
            self.start_preloading(paper_id)
        else:
            with self.output_area:
                self.output_area.clear_output()
                display(HTML(f'<p style="color: red; font-weight: bold;">Paper ID {paper_id} not found!</p>'))
    
    def display_paper(self, paper_id):
        """Display the selected paper with all its components."""
        if paper_id == self.current_paper_id:
            return  # Avoid re-rendering the same paper
        
        self.current_paper_id = paper_id
        
        with self.output_area:
            self.output_area.clear_output()
            
            # Update ID input to match current paper
            self.id_input.value = paper_id
            
            # Check if paper is in cache
            if paper_id in self.cache:
                # Use cached data (instant display)
                cached_data = self.cache[paper_id]
                summary = cached_data['summary']
                content = cached_data['content']
                review = cached_data['review']
                appendices = cached_data['appendices']
                
                # Add a small indicator that this was cached
                cache_indicator = ' <span style="color: green; font-size: 10px;">[cached]</span>'
            else:
                # Load fresh data
                summary = self.viewer.get_paper_summary(paper_id)
                content = self.viewer.get_paper_content(paper_id)
                review = self.viewer.get_review_content(paper_id)
                appendices = self.viewer.get_appendices(paper_id)
                
                # Cache it for future use
                self.cache[paper_id] = {
                    'summary': summary,
                    'content': content,
                    'review': review,
                    'appendices': appendices
                }
                cache_indicator = ''
            
            # Display paper header first (fast)
            appendix_note = f" | <strong>Appendices:</strong> {len(appendices)}" if appendices else ""
            header_html = f"""
            <div style="background-color: #f0f0f0; padding: 15px; border-radius: 5px; margin-bottom: 20px;">
                <h2 style="color: #333; margin: 0;">{summary['title']}{cache_indicator}</h2>
                <p style="margin: 10px 0;"><strong>Author:</strong> {summary['author']} | 
                   <strong>Paper ID:</strong> {summary['id']} | 
                   <strong>Tick:</strong> {summary['tick']} | 
                   <strong>Words:</strong> {summary['word_count']}{appendix_note}</p>
            </div>
            """
            display(HTML(header_html))
            
            # Display abstract
            display(HTML('<h3 style="color: #4CAF50;">Abstract</h3>'))
            abstract_html = f"""
            <div style="background-color: #e8f5e9; padding: 15px; border-left: 4px solid #4CAF50; margin-bottom: 20px;">
                {summary['abstract']}
            </div>
            """
            display(HTML(abstract_html))
            
            # Display content
            display(HTML('<h3 style="color: #2196F3;">Paper Content</h3>'))
            if content:
                display(Markdown(content))
            else:
                display(HTML('<p style="color: #999;">No content available</p>'))
            
            # Display review
            display(HTML('<h3 style="color: #FF9800;">Review</h3>'))
            if review:
                display(Markdown(review))
            else:
                display(HTML('<p style="color: #999;">No review available</p>'))

            # Display appendices
            if appendices:
                display(HTML('<h3 style="color: #9C27B0;">📎 Appendices/Addenda</h3>'))
                for i, appendix in enumerate(appendices, 1):
                    # Appendix header
                    appendix_header = f"""
                    <div style="background-color: #f3e5f5; padding: 10px; border-radius: 5px; border-left: 4px solid #9C27B0; margin: 15px 0 10px 0;">
                        <h4 style="margin: 0; color: #6a1b9a;">{appendix['title']}</h4>
                        <p style="margin: 5px 0 0 0; font-size: 12px; color: #666;">
                            <strong>Author:</strong> {appendix['author']} | 
                            <strong>Tick:</strong> {appendix['tick']} | 
                            <strong>Words:</strong> {appendix['word_count']}
                        </p>
                    </div>
                    """
                    display(HTML(appendix_header))
                    
                    # Appendix content
                    if appendix['content']:
                        display(Markdown(appendix['content']))
                    else:
                        display(HTML('<p style="color: #999; font-style: italic;">No content available</p>'))
            
    
    def __del__(self):
        """Cleanup when object is destroyed."""
        self.stop_preloading()

# Create the interactive viewer
print("Initializing interactive viewer...")
interactive_viewer = InteractivePaperViewer(viewer)
print("Interactive viewer ready!")

## Statistics Summary

In [ ]:
# Calculate and display statistics
import statistics

scores = [p['score'] for p in viewer.papers.values() if p.get('score') is not None]
reviewed_count = len(scores)
total_count = len(viewer.papers)

# Count papers with appendices
papers_with_appendices = sum(1 for pid in viewer.papers.keys() if viewer.has_appendices(pid))
total_appendices = sum(len(viewer.get_appendices(pid)) for pid in viewer.papers.keys())

stats_html = f"""
<div style="background-color: #f5f5f5; padding: 20px; border-radius: 10px;">
    <h3 style="color: #333; margin-top: 0;">Archive Statistics</h3>
    <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 15px;">
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Total Papers</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #2196F3;">{total_count}</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Reviewed Papers</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #4CAF50;">{reviewed_count}</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Papers with Appendices</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #9C27B0;">{papers_with_appendices}</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Total Appendices</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #9C27B0;">{total_appendices}</p>
        </div>
"""

if scores:
    avg_score = statistics.mean(scores)
    median_score = statistics.median(scores)
    max_score = max(scores)
    min_score = min(scores)
    
    stats_html += f"""
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Average Score</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #FF9800;">{avg_score:.2f}/10</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Median Score</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #9C27B0;">{median_score:.1f}/10</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Highest Score</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #4CAF50;">{max_score:.1f}/10</p>
        </div>
        <div style="background: white; padding: 15px; border-radius: 5px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <h4 style="margin: 0; color: #666;">Lowest Score</h4>
            <p style="font-size: 24px; margin: 5px 0; color: #f44336;">{min_score:.1f}/10</p>
        </div>
    """

stats_html += """
    </div>
</div>
"""

display(HTML(stats_html))

## Score Distribution

In [ ]:
# Create score distribution visualization
if scores:
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    ax1.hist(scores, bins=10, edgecolor='black', alpha=0.7, color='#4CAF50')
    ax1.set_xlabel('Score')
    ax1.set_ylabel('Number of Papers')
    ax1.set_title('Score Distribution')
    ax1.grid(True, alpha=0.3)
    
    # Box plot
    box = ax2.boxplot(scores, vert=False, patch_artist=True)
    box['boxes'][0].set_facecolor('#2196F3')
    ax2.set_xlabel('Score')
    ax2.set_title('Score Range and Quartiles')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No reviewed papers available for visualization")